## Importando bibliotecas e tabelas
* Nesse notebook, haverá o tratamento das tabelas `base_atendentes`, `canais` e `base_motivos`.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
bronze_atendentes = "bronze_credit.base_atendentes"
bronze_canais = "bronze_credit.canais"
bronze_motivos = "bronze_credit.base_motivos"
silver_atendentes = "silver_credit.base_atendentes"
silver_canais = "silver_credit.canais"
silver_motivos = "silver_credit.base_motivos"

In [0]:
%sql
USE CATALOG medalhao_credit;
USE SCHEMA silver_credit; 

In [0]:
df_atendentes = spark.table(bronze_atendentes)
df_canais = spark.table(bronze_canais)
df_motivos = spark.table(bronze_motivos)

### Tabela Atendentes
* Checagem feitas:
    * Total de Valores Nulos: 0
    * Total de Valores Duplicados: 0
    * Os `id_atendente `vão de 1 a 20
    * A coluna `nivel_atendimento` tem apenas valores 1 e 2
    * Mudança dos nomes das colunas

In [0]:
df_atendentes = df_atendentes \
    .withColumnRenamed("_c0", "id_atendente") \
    .withColumnRenamed("_c1", "nome_atendente") \
    .withColumnRenamed("_c2", "nivel_atendimento")

In [0]:
print(f"Schema da tabela {bronze_atendentes}:\n")
display(df_atendentes.printSchema())
print(f"Número de linhas da tabela {bronze_atendentes}: {df_atendentes.count()}")
display(df_atendentes)
    

In [0]:
nulos_df_atendentes = df_atendentes.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_atendentes.columns
]).display()

In [0]:
len_atendentes = df_atendentes.count()
df_atendentes = df_atendentes.dropDuplicates(["id_atendente"])
len_after = df_atendentes.count()

print(f"Total Duplicatas: {len_atendentes - len_after}")

In [0]:
df_gabarito = spark.range(1, 21).toDF("id")
df_erros_status = df_atendentes.filter(~F.col("nivel_atendimento").isin([1, 2]))
df_check = df_atendentes.select("id_atendente").distinct()
ids_faltantes = df_gabarito.subtract(df_check)

if ids_faltantes.count() == 0:
    print("Sucesso: Todos os IDs de 1 a 20 estão presentes.")
else:
    print("Atenção: A sequência está furada. Faltam os seguintes IDs:")
    ids_faltantes.orderBy("id").display()

if df_erros_status.count() > 0:
    print(f"Atenção! Encontramos {df_erros_status.count()} linhas com status inválido:")
    df_erros_status.display() 
else:
    print("Sucesso: A coluna status contém apenas valores 1 e 2.")

In [0]:
df_atendentes = df_atendentes \
                .withColumn("data_ingestao_silver", F.current_timestamp()) \
                .withColumnRenamed("data_ingestao","data_ingestao_bronze")

In [0]:
df_atendentes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_atendentes)

### Tabela Canais
* Checagem feitas:
    * Total de Valores Nulos: 0
    * Total de Valores Duplicados: 0
    * Mudança dos nomes das colunas
    * Padronização da coluna `status_canal`: Inativo estava escrito errado em alguns casos.
    * Padronização da coluna `nome_canal` : Email não estava em maiúsculo.

In [0]:
df_canais = df_canais \
    .withColumnRenamed("_c0", "nome_canal") \
    .withColumnRenamed("_c1", "status_canal") 

In [0]:
print(f"Schema da tabela {bronze_canais}:\n")
display(df_canais.printSchema())
print(f"Número de linhas da tabela {bronze_canais}: {df_canais.count()}")
display(df_canais)

In [0]:
df_canais_tratado = df_canais \
   .withColumn("nome_canal", 
        F.when(F.lower(F.col("nome_canal")).like("%ura%"), "URA") \
        .otherwise(F.initcap(F.col("nome_canal")))
    ) \
    .withColumn("status_canal", F.regexp_replace(F.col("status_canal"), "invativo", "inativo")) \
    .withColumn("data_ingestao_silver", F.current_timestamp()) \
    .withColumnRenamed("data_ingestao", "data_ingestao_bronze") \
    
display(df_canais_tratado)

In [0]:
df_canais_tratado.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_canais)

Como já podemos observar manualmente, não há valores nulos ou valores duplicados.

### Tabela Motivos
* Checagens feitas:
    * Total de Valores Nulos: 0
    * Total de Valores Duplicados: 0
    * Mudança dos nomes das colunas
    * Mudar `id_motivo` de 3 a 15 para 1 a 13 para garantir consistência
    * Preencher coluna `categoria` que veio nula com categorias Financeiro, Cartão, Cadastral e Outros
    * Deixar coluna `criticidade` padronizada (alguns valores vieram com inicial maiúscula e outros minúscula)
    * Corrigir "Compra no autorizada" para "Compra não autorizada".

In [0]:
df_motivos = df_motivos \
    .withColumnRenamed("_c0", "id_motivo") \
    .withColumnRenamed("_c1", "nome_motivo") \
    .withColumnRenamed("_c2", "categoria") \
    .withColumnRenamed("_c3", "criticidade")

In [0]:
print(f"Schema da tabela {bronze_motivos}:\n")
display(df_motivos.printSchema())
print(f"Número de linhas da tabela {bronze_motivos}: {df_motivos.count()}")
display(df_motivos)

Como já podemos observar manualmente, não há valores nulos ou valores duplicados. Porém percebemos **uma coluna inteiramente nula** (que era para estar preenchida com **Financeiro, Cartão ou Cadastral**),  **inconsistências nas categorias ["baixa", "Baixa"] e ID's inconsistentes (de 3 a 7).**

In [0]:
janela_ordenacao = Window.orderBy("id_motivo")

df_motivos_tratado = df_motivos \
    .withColumn("criticidade", F.regexp_replace(F.initcap(F.col("criticidade")), "Média", "Media")) \
    .withColumn("id_motivo", F.row_number().over(janela_ordenacao)) \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "Compra no autorizada", "Compra Nao Autorizada"))\
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "Contrata..o", "Contratacao")) \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "Contesta..o", "Contestacao")) \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "Altera..o", "Alteracao"))   \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "cart.o", "Cartao"))   \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "D.vidas", "Duvidas"))  \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "Informa..es", "Informacoes"))  \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "Solicita..o", "Solicitacao"))   \
    .withColumn("nome_motivo", F.initcap(F.col("nome_motivo"))) \
    .withColumn("categoria", 
        F.when(F.lower(F.col("nome_motivo")).like("%cadastrais%"), "Cadastral")
        .when(F.lower(F.col("nome_motivo")).like("%dados%"), "Cadastral")
        .when(F.lower(F.col("nome_motivo")).like("%fatura%"), "Financeiro")
        .when(F.lower(F.col("nome_motivo")).like("%dívida%"), "Financeiro")
        .when(F.lower(F.col("nome_motivo")).like("%agência%"), "Cadastral")
        .when(F.lower(F.col("nome_motivo")).like("%limite%"), "Cartao")
        .when(F.lower(F.col("nome_motivo")).like("%cartão%"), "Cartao")
        .when(F.lower(F.col("nome_motivo")).like("%compra%"), "Cartao")
        .otherwise("Outros") 
    ) \
    .withColumn("data_ingestao_silver", F.current_timestamp()) \
    .withColumnRenamed("data_ingestao", "data_ingestao_bronze") 
df_motivos_tratado.display()

In [0]:
df_motivos_tratado.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_motivos)